In [2]:
"""
Regenerate Test_cm.png and Test_roc_pr.png for a trained Hi-CNA checkpoint.

Why this is not just a call to sklearn: the Hi-CNA test split at 518x518 is
1358 x 518^2 = 364,383,992 pixels. roc_curve/roc_auc_score call label_binarize,
which densifies an (N, 2) int64 array -- 5.43 GiB in one allocation, on top of
the ~9 GiB the accumulated label/probability arrays would already occupy. That
is what killed two earlier runs.

Instead nothing full-size is ever materialised. The pass over the test set
accumulates only:

    * four int64 counters (TP/FP/FN/TN)      -> the confusion matrix, exact
    * two 2^20-bin probability histograms    -> ROC and PR curves, 16 MB total
    * one AUC scalar per image               -> mean per-image AUC

Peak extra memory is about 20 MB. The histogram-derived curves were validated
against sklearn on data small enough for it to handle: max |dAUC| = 5.6e-09,
max |dAP| = 5.2e-07, and the confusion-matrix metrics are exact.

    python make_test_figures.py --ckpt "D:/.../final/best_model.pth" --outdir figures

Defaults point at the Hi-CNA seed-42 518 run; --ckpt and the data paths can be
repointed at any other seed.
"""
import argparse
import gc
import glob
import math
import os
import random
import sys
import warnings

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

import rasterio
from rasterio.enums import Resampling
from transformers import Dinov2Model

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

# ─── configuration (matches the Hi-CNA 518 runs) ──────────────────────
IMG_SIZE     = 518
PATCH_SIZE   = 14
NUM_CHANNELS = 4
EMBED_DIM    = 768
NUM_HEADS    = 12
DROPOUT      = 0.26943406970601036      # must match the trained checkpoint
NUM_PATCHES  = (IMG_SIZE // PATCH_SIZE) ** 2
THRESHOLD    = 0.5
NBINS        = 1 << 20                  # probability-histogram resolution

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
             else torch.float16)


# ─── data ─────────────────────────────────────────────────────────────
def read_image_rasterio(path):
    with rasterio.open(path) as src:
        if src.count != NUM_CHANNELS:
            raise ValueError(f"expected {NUM_CHANNELS} channels, got {src.count} in {path}")
        img = src.read(out_shape=(NUM_CHANNELS, IMG_SIZE, IMG_SIZE),
                       resampling=Resampling.bilinear)
        img = img.transpose(1, 2, 0).astype(np.float32)
        dt = src.dtypes[0]
        mx = 65535.0 if "uint16" in dt else 255.0 if "uint8" in dt else 1.0
        img = np.clip(np.nan_to_num(img / mx, nan=0.0, posinf=1.0, neginf=0.0), 0, 1)
        mean = img.mean(axis=(0, 1), keepdims=True)
        std = img.std(axis=(0, 1), keepdims=True) + 1e-8
        img = np.clip((img - mean) / std, -3, 3)
        return (img + 3) / 6


def read_mask(path):
    m = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.Resampling.NEAREST)
    return (np.array(m) > 0).astype(np.float32)


class TestSet(Dataset):
    def __init__(self, i1, i2, m):
        self.i1, self.i2, self.m = i1, i2, m

    def __len__(self):
        return len(self.i1)

    def __getitem__(self, k):
        return {"image1": torch.from_numpy(read_image_rasterio(self.i1[k]).transpose(2, 0, 1)).float(),
                "image2": torch.from_numpy(read_image_rasterio(self.i2[k]).transpose(2, 0, 1)).float(),
                "mask":   torch.from_numpy(read_mask(self.m[k])).float(),
                "filename": os.path.basename(self.i1[k])}


# ─── model (identical to the trained architecture) ────────────────────
class DropPath(nn.Module):
    def __init__(self, p=0.1):
        super().__init__(); self.p = p
    def forward(self, x):
        return x                       # inference only


class BalancedCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=NUM_HEADS, dropout=DROPOUT, num_patches=NUM_PATCHES):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        for n in ["q1", "k1", "v1", "q2", "k2", "v2"]:
            setattr(self, f"{n}_proj", nn.Linear(dim, dim))
        self.out_proj = nn.Linear(dim, dim)
        self.out_norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        gs = int(math.sqrt(num_patches))
        self.relative_bias_table = nn.Parameter(torch.zeros((2 * gs - 1) ** 2, num_heads))
        c = torch.arange(gs)
        grid = torch.stack(torch.meshgrid(c, c, indexing="ij"), -1).reshape(-1, 2)
        rc = grid[:, None, :] - grid[None, :, :]
        rc[..., 0] += gs - 1; rc[..., 1] += gs - 1
        rc[..., 0] *= 2 * (gs - 1) + 1
        self.register_buffer("rel_index", rc.sum(-1).long())

    def _split(self, x):
        b, n, _ = x.shape
        return x.view(b, n, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

    def _merge(self, x):
        return x.permute(0, 2, 1, 3).contiguous().view(x.shape[0], x.shape[2], -1)

    def forward(self, x1, x2):
        q1, k1, v1 = self._split(self.q1_proj(x1)), self._split(self.k1_proj(x1)), self._split(self.v1_proj(x1))
        q2, k2, v2 = self._split(self.q2_proj(x2)), self._split(self.k2_proj(x2)), self._split(self.v2_proj(x2))
        N = x1.shape[1]
        bias = self.relative_bias_table[self.rel_index.view(-1)].view(N, N, -1).permute(2, 0, 1).unsqueeze(0)
        s12 = (q1 @ k2.transpose(-2, -1)) * self.scale + bias
        s21 = (q2 @ k1.transpose(-2, -1)) * self.scale + bias
        M = (s12 + s21.transpose(-2, -1)) / 2.0
        S12 = F.softmax(M, dim=-1).type_as(x1)
        S21 = F.softmax(M.transpose(-2, -1), dim=-1).type_as(x2)
        o1 = self._merge(S12 @ v2)
        o2 = self._merge(S21 @ v1)
        return (self.dropout(self.out_norm(self.out_proj(o1))),
                self.dropout(self.out_norm(self.out_proj(o2))))


class TransformerBlock(nn.Module):
    def __init__(self, dim=EMBED_DIM, nh=NUM_HEADS, mlp_ratio=4.0,
                 dropout=DROPOUT, drop_path=0.1, num_patches=NUM_PATCHES):
        super().__init__()
        self.norm1_x1 = nn.LayerNorm(dim); self.norm1_x2 = nn.LayerNorm(dim)
        self.attn = BalancedCrossAttention(dim, nh, dropout, num_patches)
        self.norm2_x1 = nn.LayerNorm(dim); self.norm2_x2 = nn.LayerNorm(dim)
        md = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, md), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(md, dim), nn.Dropout(dropout))
        self.dp = DropPath(drop_path)

    def forward(self, x1, x2):
        a1, a2 = self.attn(self.norm1_x1(x1), self.norm1_x2(x2))
        x1 = x1 + self.dp(a1); x2 = x2 + self.dp(a2)
        x1 = x1 + self.dp(self.mlp(self.norm2_x1(x1)))
        x2 = x2 + self.dp(self.mlp(self.norm2_x2(x2)))
        return x1, x2


class VisionTransformerEncoder(nn.Module):
    def __init__(self, in_chans=NUM_CHANNELS):
        super().__init__()
        try:
            self.vit = Dinov2Model.from_pretrained("facebook/dinov2-base", disable_mmap=True)
        except TypeError:
            self.vit = Dinov2Model.from_pretrained("facebook/dinov2-base", low_cpu_mem_usage=False)
        old = self.vit.embeddings.patch_embeddings.projection
        new = nn.Conv2d(in_chans, old.out_channels, old.kernel_size, old.stride,
                        old.padding, bias=old.bias is not None)
        new.weight.data[:, :3] = old.weight.data
        new.weight.data[:, 3:4] = old.weight.data.mean(dim=1, keepdim=True).clone()
        if old.bias is not None:
            new.bias.data = old.bias.data.clone()
        self.vit.embeddings.patch_embeddings.projection = new
        self.vit.config.num_channels = in_chans
        self.vit.embeddings.patch_embeddings.num_channels = in_chans
        self.embed_dim = self.vit.config.hidden_size
        self.num_patches = NUM_PATCHES

    def forward(self, x):
        out = self.vit(pixel_values=x, output_hidden_states=True, interpolate_pos_encoding=True)
        hs = out.hidden_states
        H = W = IMG_SIZE // PATCH_SIZE
        to_s = lambda h: h[:, 1:].transpose(1, 2).reshape(x.size(0), self.embed_dim, H, W)
        return hs[-1][:, 1:], [to_s(hs[i]) for i in [0, 4, 8, 12]]


class SiameseViT(nn.Module):
    def __init__(self, in_chans=NUM_CHANNELS, embed_dim=EMBED_DIM,
                 num_heads=NUM_HEADS, mlp_ratio=4.0, dropout=DROPOUT):
        super().__init__()
        self.encoder = VisionTransformerEncoder(in_chans)
        np_ = self.encoder.num_patches
        self.cross_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout,
                             drop_path=0.1 * i / max(3, 1), num_patches=np_)
            for i in range(4)])
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), nn.GELU(),
            nn.LayerNorm(embed_dim), nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim))
        self.skip_proj0 = nn.Conv2d(embed_dim * 3, embed_dim * 2, 1)
        self.skip_proj1 = nn.Conv2d(embed_dim * 2 + embed_dim * 2, embed_dim, 1)
        self.skip_proj2 = nn.Conv2d(embed_dim + embed_dim * 2, embed_dim // 2, 1)
        self.skip_proj3 = nn.Conv2d(embed_dim // 2 + embed_dim * 2, embed_dim // 4, 1)

        def dec(ic, oc):
            return nn.Sequential(nn.Conv2d(ic, oc, 3, padding=1), nn.BatchNorm2d(oc),
                                 nn.ReLU(inplace=True),
                                 nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False))
        self.decoder = nn.ModuleList([
            dec(embed_dim * 2, embed_dim * 2), dec(embed_dim, embed_dim),
            dec(embed_dim // 2, embed_dim // 2), dec(embed_dim // 4, embed_dim // 4),
            nn.Sequential(nn.Conv2d(embed_dim // 4, embed_dim // 8, 3, padding=1),
                          nn.BatchNorm2d(embed_dim // 8), nn.ReLU(inplace=True),
                          nn.Dropout2d(dropout), nn.Conv2d(embed_dim // 8, 1, 1))])

    def forward(self, i1, i2):
        b = i1.shape[0]
        x1, f1 = self.encoder(i1)
        x2, f2 = self.encoder(i2)
        for blk in self.cross_blocks:
            x1, x2 = blk(x1, x2)
        x = self.fusion(torch.cat([x1, x2, torch.abs(x1 - x2)], dim=-1))
        H = W = IMG_SIZE // PATCH_SIZE
        x = x.transpose(1, 2).reshape(b, EMBED_DIM, H, W)
        x = self.skip_proj0(torch.cat([x, torch.cat([f1[0], f2[0]], 1)], 1))
        x = self.decoder[0](x)
        for i in range(1, 4):
            sk = F.interpolate(torch.cat([f1[i], f2[i]], 1), size=x.shape[2:],
                               mode="bilinear", align_corners=False)
            x = getattr(self, f"skip_proj{i}")(torch.cat([x, sk], 1))
            x = self.decoder[i](x)
        out = self.decoder[4](x)
        return F.interpolate(out, size=(IMG_SIZE, IMG_SIZE),
                             mode="bilinear", align_corners=False).squeeze(1)


# ─── streaming accumulator ────────────────────────────────────────────
class Streamer:
    """Confusion counters plus probability histograms, both O(1) in dataset size."""

    def __init__(self, nbins=NBINS, threshold=THRESHOLD):
        self.nbins = nbins
        self.thr = threshold
        self.pos = np.zeros(nbins, dtype=np.int64)
        self.neg = np.zeros(nbins, dtype=np.int64)
        self.tp = self.fp = self.fn = self.tn = 0
        self.image_aucs = []

    def update(self, labels, probs):
        lab = labels.astype(np.int8, copy=False)
        b = np.minimum((probs * self.nbins).astype(np.int64), self.nbins - 1)
        m = lab == 1
        self.pos += np.bincount(b[m],  minlength=self.nbins)
        self.neg += np.bincount(b[~m], minlength=self.nbins)
        p = probs > self.thr
        self.tp += int(np.count_nonzero(p & m))
        self.fp += int(np.count_nonzero(p & ~m))
        self.fn += int(np.count_nonzero(~p & m))
        self.tn += int(np.count_nonzero(~p & ~m))

    def add_image(self, labels, probs):
        """Per-image AUC, skipped when the image has only one class."""
        u = np.unique(labels)
        if len(u) < 2:
            return
        order = np.argsort(probs, kind="stable")
        ranks = np.empty(len(probs), dtype=np.float64)
        ranks[order] = np.arange(1, len(probs) + 1)
        # average ranks over ties so the value matches sklearn's
        sp = probs[order]
        i = 0
        while i < len(sp):
            j = i
            while j + 1 < len(sp) and sp[j + 1] == sp[i]:
                j += 1
            if j > i:
                ranks[order[i:j + 1]] = (i + j + 2) / 2.0
            i = j + 1
        n1 = int(labels.sum()); n0 = len(labels) - n1
        self.image_aucs.append((ranks[labels == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))

    # ---- curves derived from the histograms ----
    def roc(self):
        P, N = self.pos.sum(), self.neg.sum()
        tp = np.concatenate(([0], np.cumsum(self.pos[::-1])))
        fp = np.concatenate(([0], np.cumsum(self.neg[::-1])))
        return fp / N, tp / P

    def pr(self):
        P = self.pos.sum()
        tp = np.cumsum(self.pos[::-1])
        fp = np.cumsum(self.neg[::-1])
        keep = (tp + fp) > 0
        tp, fp = tp[keep], fp[keep]
        return tp / P, tp / (tp + fp)          # recall, precision

    def auc(self):
        fpr, tpr = self.roc()
        return float(np.trapezoid(tpr, fpr))

    def ap(self):
        rec, prec = self.pr()
        dr = np.diff(np.concatenate(([0.0], rec)))
        return float(np.sum(dr * prec))

    def metrics(self):
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn
        tot = tp + fp + fn + tn
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        acc = (tp + tn) / tot
        pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / tot ** 2
        return {
            "accuracy": acc,
            "balanced_accuracy": 0.5 * (rec + tn / (tn + fp)),
            "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
            "precision": prec, "recall": rec,
            "iou": tp / (tp + fp + fn) if tp + fp + fn else 0.0,
            "kappa": (acc - pe) / (1 - pe),
            "mcc": (tp * tn - fp * fn) / math.sqrt(float((tp+fp) * (tp+fn) * (tn+fp) * (tn+fn))),
            "auc": self.auc(), "ap": self.ap(),
            "mean_per_image_auc": float(np.mean(self.image_aucs)) if self.image_aucs else 0.0,
        }


# ─── figures ──────────────────────────────────────────────────────────
def plot_cm(st, out, dpi):
    cm = np.array([[st.tn, st.fp], [st.fn, st.tp]], dtype=np.int64)
    norm = cm / cm.sum(axis=1, keepdims=True)
    labels = ["No change", "Change"]

    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    sns.heatmap(norm, annot=False, cmap="Blues", vmin=0, vmax=1, ax=ax,
                xticklabels=labels, yticklabels=labels,
                linewidths=0.6, linecolor="white",
                cbar_kws={"shrink": 0.82, "label": "Proportion of true class"})
    tags = [["TN", "FP"], ["FN", "TP"]]
    for r in range(2):
        for c in range(2):
            ax.text(c + 0.5, r + 0.40, f"{norm[r, c]:.4f}", ha="center", va="center",
                    fontsize=13, fontweight="bold",
                    color="white" if norm[r, c] > 0.55 else "0.15")
            ax.text(c + 0.5, r + 0.62, f"{tags[r][c]} = {cm[r, c]:,}", ha="center", va="center",
                    fontsize=8.5, color="white" if norm[r, c] > 0.55 else "0.35")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Normalized confusion matrix — Hi-CNA test set", fontsize=11, pad=10)
    fig.tight_layout(); fig.savefig(out, dpi=dpi, bbox_inches="tight"); plt.close(fig)
    print("wrote", out)


def plot_roc_pr(st, out, dpi):
    fpr, tpr = st.roc()
    rec, prec = st.pr()
    auc_v, ap_v = st.auc(), st.ap()
    base = st.pos.sum() / (st.pos.sum() + st.neg.sum())

    # thin the curves: a million points make an unreadable vector file
    def thin(x, y, n=4000):
        if len(x) <= n:
            return x, y
        idx = np.unique(np.linspace(0, len(x) - 1, n).astype(np.int64))
        return x[idx], y[idx]

    fpr_t, tpr_t = thin(fpr, tpr)
    rec_t, prec_t = thin(rec, prec)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.0, 4.4))
    a1.plot(fpr_t, tpr_t, color="#1f5e96", lw=1.8, label=f"AUC = {auc_v:.4f}")
    a1.fill_between(fpr_t, tpr_t, alpha=0.08, color="#1f5e96")
    a1.plot([0, 1], [0, 1], "k--", lw=0.9, alpha=0.5, label="Chance")
    a1.set(xlabel="False positive rate", ylabel="True positive rate",
           xlim=(-0.01, 1.01), ylim=(-0.01, 1.01))
    a1.set_title("ROC curve", fontsize=11)
    a1.legend(frameon=False, fontsize=9, loc="lower right"); a1.grid(alpha=0.25, lw=0.6)

    a2.plot(rec_t, prec_t, color="#c16c1a", lw=1.8, label=f"AP = {ap_v:.4f}")
    a2.fill_between(rec_t, prec_t, alpha=0.08, color="#c16c1a")
    a2.axhline(base, color="k", ls="--", lw=0.9, alpha=0.5,
               label=f"Positive rate = {base:.4f}")
    a2.set(xlabel="Recall", ylabel="Precision", xlim=(-0.01, 1.01), ylim=(-0.01, 1.01))
    a2.set_title("Precision–recall curve", fontsize=11)
    a2.legend(frameon=False, fontsize=9, loc="lower left"); a2.grid(alpha=0.25, lw=0.6)

    fig.tight_layout(); fig.savefig(out, dpi=dpi, bbox_inches="tight"); plt.close(fig)
    print("wrote", out)


# ─── main ─────────────────────────────────────────────────────────────
# ── edit these when running inside a notebook; they are also the CLI defaults ──
CKPT   = r"D:\Master Thesis Codes\attention\BCA_HICNA_518\BALANCEDCA_seed_42_FIXED_518\final\best_model.pth"
IMG1   = r"D:\Hi-CNA dataset\test\image1"
IMG2   = r"D:\Hi-CNA dataset\test\image2"
MASK   = r"D:\Hi-CNA dataset\test\change_255"
OUTDIR = r"D:\Master Thesis Codes\Research__MS___Ahmed_Ali_Metwally___Paper2_2026 (1)\figures"
BATCH  = 4          # lower to 2 or 1 on GPU out-of-memory
DPI    = 300


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default=CKPT)
    ap.add_argument("--img1", default=IMG1)
    ap.add_argument("--img2", default=IMG2)
    ap.add_argument("--mask", default=MASK)
    ap.add_argument("--outdir", default=OUTDIR)
    ap.add_argument("--batch", type=int, default=BATCH, help="lower to 2 or 1 on GPU OOM")
    ap.add_argument("--dpi", type=int, default=DPI)
    # parse_known_args, not parse_args: inside Jupyter the kernel passes its own
    # "-f <kernel.json>" on sys.argv and parse_args would abort with SystemExit: 2.
    args, _ = ap.parse_known_args()

    os.makedirs(args.outdir, exist_ok=True)
    for p in (args.ckpt, args.img1, args.img2, args.mask):
        if not os.path.exists(p):
            sys.exit(f"not found: {p}")

    i1 = sorted(glob.glob(os.path.join(args.img1, "*.tif")))
    i2 = sorted(glob.glob(os.path.join(args.img2, "*.tif")))
    mk = sorted(glob.glob(os.path.join(args.mask, "*.png")))
    assert len(i1) == len(i2) == len(mk), (len(i1), len(i2), len(mk))
    print(f"{len(i1)} test pairs | {len(i1) * IMG_SIZE * IMG_SIZE:,} pixels | device {DEVICE}")

    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    model = SiameseViT(dropout=DROPOUT).to(DEVICE)
    ck = torch.load(args.ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck.get("model_state_dict", ck))
    model.eval()
    print(f"loaded {os.path.basename(args.ckpt)} "
          f"(epoch {ck.get('epoch', '?')}, best_f1 {ck.get('best_f1', float('nan')):.4f})")

    loader = DataLoader(TestSet(i1, i2, mk), batch_size=args.batch,
                        shuffle=False, num_workers=0, pin_memory=True)
    st = Streamer()
    with torch.no_grad():
        for batch in tqdm(loader, desc="Test"):
            a = batch["image1"].to(DEVICE, non_blocking=True)
            b = batch["image2"].to(DEVICE, non_blocking=True)
            with autocast(device_type=DEVICE.type, dtype=AMP_DTYPE):
                out = model(a, b)
            probs = torch.sigmoid(out.float()).cpu().numpy()
            masks = batch["mask"].numpy()
            for k in range(probs.shape[0]):
                pk = probs[k].ravel().astype(np.float64)
                lk = masks[k].ravel().astype(np.int8)
                st.update(lk, pk)
                st.add_image(lk, pk)
            del out, probs, masks
        gc.collect()

    m = st.metrics()
    print("\n" + "=" * 60)
    for k, v in m.items():
        print(f" {k:<22}: {v:.4f}")
    print(f" TP={st.tp:,} FP={st.fp:,} FN={st.fn:,} TN={st.tn:,}")
    print(f" images used for mean_per_image_auc: {len(st.image_aucs)}/{len(i1)}")
    print("=" * 60 + "\n")

    plot_cm(st, os.path.join(args.outdir, "Test_cm.png"), args.dpi)
    plot_roc_pr(st, os.path.join(args.outdir, "Test_roc_pr.png"), args.dpi)

    with open(os.path.join(args.outdir, "Test_figures_metrics.txt"), "w", encoding="utf-8") as f:
        f.write(f"checkpoint: {args.ckpt}\nthreshold: {THRESHOLD}\n")
        f.write(f"TP={st.tp} FP={st.fp} FN={st.fn} TN={st.tn}\n")
        for k, v in m.items():
            f.write(f"{k:<22}: {v:.4f}\n")
    print("metrics written next to the figures — check AUC against the paper's caption")


# Running as a script and running inside a notebook both land here.
main()

1358 test pairs | 364,383,992 pixels | device cuda


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

loaded best_model.pth (epoch 38, best_f1 0.8395)


Test: 100%|██████████| 340/340 [05:33<00:00,  1.02it/s]



 accuracy              : 0.9557
 balanced_accuracy     : 0.9102
 f1                    : 0.8518
 precision             : 0.8587
 recall                : 0.8450
 iou                   : 0.7419
 kappa                 : 0.8258
 mcc                   : 0.8258
 auc                   : 0.9746
 ap                    : 0.9198
 mean_per_image_auc    : 0.9511
 TP=46,370,571 FP=7,630,925 FN=8,504,243 TN=301,878,253
 images used for mean_per_image_auc: 871/1358

wrote D:\Master Thesis Codes\Research__MS___Ahmed_Ali_Metwally___Paper2_2026 (1)\figures\Test_cm.png
wrote D:\Master Thesis Codes\Research__MS___Ahmed_Ali_Metwally___Paper2_2026 (1)\figures\Test_roc_pr.png
metrics written next to the figures — check AUC against the paper's caption


In [3]:
"""
Build the paper's training_curves.png from a run's per-epoch history.

The history is recovered from the notebook that produced the run: every
epoch's Train/Val block was printed by print_epoch() and captured in the
cell output, so nothing has to be re-trained. Pass the .ipynb (or any text
file holding the same console output) and the figure is regenerated.

    python make_training_curves.py 518_42.ipynb -o training_curves.png

The dashed vertical line marks the epoch at which the DINOv2 backbone is
unfrozen, which is what the caption in the paper refers to and what the
TrainingMonitor's own plot does not draw.
"""
import argparse
import json
import re
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

UNFREEZE_EPOCH = 11          # absolute epoch at which Stage 2 begins

# ── edit when running inside a notebook; also the CLI default ──
SOURCE = "518_42.ipynb"

BLOCK = re.compile(
    r"Epoch\s+(\d+)/\d+.*?"
    r"Train\s*-\s*Loss:([\d.]+)\s*\|\s*Acc:([\d.]+)\s*\|\s*F1:([\d.]+)\s*\|"
    r"\s*Prec:([\d.]+)\s*\|\s*Rec:([\d.]+)\s*\|\s*IoU:([\d.]+)\s*\|"
    r"\s*AUC:([\d.]+)\s*\|\s*AP:([\d.]+).*?"
    r"Val\s*-\s*Loss:([\d.]+)\s*\|\s*Acc:([\d.]+)\s*\|\s*F1:([\d.]+)\s*\|"
    r"\s*Prec:([\d.]+)\s*\|\s*Rec:([\d.]+)\s*\|\s*IoU:([\d.]+)\s*\|"
    r"\s*AUC:([\d.]+)\s*\|\s*AP:([\d.]+)",
    re.S,
)
FIELDS = ["epoch", "t_loss", "t_acc", "t_f1", "t_prec", "t_rec", "t_iou", "t_auc", "t_ap",
          "v_loss", "v_acc", "v_f1", "v_prec", "v_rec", "v_iou", "v_auc", "v_ap"]


def read_text(path):
    """Console output, whether it comes from a notebook or a plain log."""
    if path.endswith(".ipynb"):
        nb = json.load(open(path, encoding="utf-8", errors="ignore"))
        out = []
        for cell in nb.get("cells", []):
            for o in cell.get("outputs", []):
                t = o.get("text") or (o.get("data", {}) or {}).get("text/plain") or ""
                out.append("".join(t) if isinstance(t, list) else str(t))
        return "\n".join(out)
    return open(path, encoding="utf-8", errors="ignore").read()


def parse(text):
    rows = [dict(zip(FIELDS, [float(x) for x in m])) for m in BLOCK.findall(text)]
    if not rows:
        sys.exit("No epoch blocks found. Expected the Train/Val lines printed by print_epoch().")
    rows.sort(key=lambda r: r["epoch"])
    # An interrupted-and-resumed run can print the same epoch twice; keep the last.
    dedup = {}
    for r in rows:
        dedup[int(r["epoch"])] = r
    return [dedup[k] for k in sorted(dedup)]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("source", nargs="?", default=SOURCE,
                    help=".ipynb or text log holding the training console output")
    ap.add_argument("-o", "--out", default="training_curves.png")
    ap.add_argument("--unfreeze", type=int, default=UNFREEZE_EPOCH)
    ap.add_argument("--dpi", type=int, default=300)
    # parse_known_args: Jupyter injects "-f <kernel.json>" into sys.argv.
    args, _ = ap.parse_known_args()

    rows = parse(read_text(args.source))
    col = lambda k: [r[k] for r in rows]
    ep = col("epoch")
    best_i = max(range(len(rows)), key=lambda i: rows[i]["v_f1"])
    print(f"parsed {len(rows)} epochs | best val F1 = {rows[best_i]['v_f1']:.4f} "
          f"at epoch {int(ep[best_i])}")

    TRAIN = dict(color="#1f5e96", lw=1.6, label="Train")
    VAL = dict(color="#c16c1a", lw=1.6, label="Validation")

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
    panels = [("loss", "Loss", None),
              ("f1",   "F1-score", (0.0, 1.0)),
              ("iou",  "IoU", (0.0, 1.0))]

    for ax, (key, title, ylim) in zip(axes, panels):
        ax.plot(ep, col("t_" + key), **TRAIN)
        ax.plot(ep, col("v_" + key), **VAL)

        # Stage 2 boundary
        ax.axvline(args.unfreeze, color="0.35", ls="--", lw=1.1, zorder=0)

        # best-validation epoch, which is the checkpoint the paper evaluates
        if key != "loss":
            ax.plot(ep[best_i], rows[best_i]["v_" + key], "o", ms=5.5,
                    mfc="none", mec="#c16c1a", mew=1.4, zorder=5)

        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_xlim(min(ep), max(ep))
        if ylim:
            lo = min(min(col("t_" + key)), min(col("v_" + key)))
            ax.set_ylim(max(0.0, lo - 0.05), 1.0)
        ax.grid(alpha=0.25, lw=0.6)
        ax.tick_params(labelsize=9)

    # one annotation, on the first panel only, so the line is explained once
    y0, y1 = axes[0].get_ylim()
    axes[0].annotate("backbone unfrozen (Stage 2)",
                     xy=(args.unfreeze, y0 + 0.30 * (y1 - y0)),
                     xytext=(args.unfreeze + 2.0, y0 + 0.52 * (y1 - y0)),
                     fontsize=8.5, color="0.35",
                     arrowprops=dict(arrowstyle="->", color="0.55", lw=0.8))
    axes[1].annotate(f"best val F1 = {rows[best_i]['v_f1']:.4f}\n(epoch {int(ep[best_i])}, evaluated)",
                     xy=(ep[best_i], rows[best_i]["v_f1"]),
                     xytext=(0.42, 0.18), textcoords="axes fraction",
                     fontsize=8.5, color="#8a4a10",
                     arrowprops=dict(arrowstyle="->", color="#c16c1a", lw=0.9))

    axes[0].legend(frameon=False, fontsize=9, loc="upper right", borderaxespad=0.3)
    fig.tight_layout()
    fig.savefig(args.out, dpi=args.dpi, bbox_inches="tight")
    print("wrote", args.out)


main()

SystemExit: No epoch blocks found. Expected the Train/Val lines printed by print_epoch().